<div style="border-left:6px solid #00897B;padding:12px 18px;background:#F5F5F5">
<span style="color:#00897B;font-weight:bold">PAUL MINCHELLA · ONLINE LEARNING · TEMPLATE ÉTUDIANT</span>
<h1 style="color:#283593">Apprendre un jour à la fois</h1>
<b>OGD, OGD + Ridge et perte newsvendor sur un flux réel de demande</b><br>
Questions, fonctions à compléter et réponses rédigées · Durée suggérée : 2h30
</div>

Ce TP accompagne [les slides du cours](../Cours_Online_Learning.pdf). On prévoit le nombre journalier
de vélos loués à Washington (Capital Bikeshare) et l'on apprend **après**, jamais avant, la révélation
de chaque observation. Le jeu de données est assez petit pour écrire chaque algorithme à la main,
sans cacher les mises à jour dans une bibliothèque.

<div style="border-left:4px solid #00897B;padding:8px 14px;background:#E0F2F1">
<b style="color:#283593">Mode d'emploi</b><br>

- Complétez le corps des fonctions et des classes marquées <code># À compléter</code>. Leurs <b>noms, signatures et docstrings sont imposés</b> :
  la suite du notebook les appelle.
- Les cellules de travail contiennent des commentaires qui indiquent quoi calculer et dans quelles variables ranger le résultat.
  Respectez ces noms (<code>X_train</code>, <code>results</code>, <code>scorecard</code>…), car les cellules suivantes les réutilisent.
- Rédigez vos réponses dans les cellules <b>Votre réponse :</b>, en quelques phrases.
- Ne choisissez <b>jamais</b> un hyperparamètre en regardant le test (2012).
- Avant de rendre le notebook : <i>Kernel → Restart & Run All</i>, et vérifiez qu'il s'exécute sans erreur.
</div>

### Objectifs

À la fin du TP, vous saurez :

1. décrire une série temporelle et repérer les variables disponibles **avant** la décision ;
2. vérifier les hypothèses des garanties de regret (domaine compact, covariables et cible bornées) ;
3. implémenter OGD projeté pour la perte quadratique, avec et sans ridge ;
4. implémenter OGD pour la perte newsvendor et constater que l'optimum est le quantile critique $\tau = c_b/(c_b+c_h)$ ;
5. évaluer proprement (train puis test, prédire avant d'apprendre) avec le RMSE et le regret moyen.

### Plan

| Partie | Contenu | Slides |
|:--|:--|:--|
| **0** | Packages et données | — |
| **1** | Statistiques descriptives | 9–14 |
| **2** | Apprentissage en ligne : hypothèses, covariables, protocole, OGD, Ridge, Newsvendor | protocole 15–25 ; regret 26–29 ; OGD 44–49 ; régression 53–59 ; stock 60–64 |
| **3** | Métriques finales : RMSE, regret moyen, conclusion | 26–29, 65 |

<div style="border-left:4px solid #42A5F5;padding:8px 14px;background:#E3F2FD">
<b style="color:#283593">Rappel — moyenne mobile</b><br>
Une moyenne mobile de fenêtre $w$ remplace chaque valeur par la moyenne de $w$ valeurs voisines. Elle lisse le bruit
journalier pour faire apparaître la tendance et la saisonnalité.

- Moyenne mobile <b>passée</b> (causale), qui n'utilise que le passé :
$\displaystyle \bar y_t^{\,\text{passée}} = \frac1w \sum_{k=0}^{w-1} y_{t-k}.$
- Moyenne mobile <b>centrée</b>, qui utilise autant de jours après $t$ qu'avant :
$\displaystyle \bar y_t^{\,\text{centrée}} = \frac1w \sum_{k=-\lfloor w/2 \rfloor}^{\lceil w/2 \rceil - 1} y_{t+k}.$

La version centrée est mieux alignée sur la série : c'est le bon outil pour la <i>décrire</i>. Elle regarde le futur,
elle ne doit donc <b>jamais</b> servir à <i>prévoir</i>. En pandas : <code>s.rolling(w).mean()</code> (passée) et
<code>s.rolling(w, center=True).mean()</code> (centrée).
</div>

### Nom, prénom et date

...

# Partie 0 — Packages et données

On n'importe que le nécessaire : NumPy pour l'algèbre, pandas pour le tableau, Matplotlib pour les
figures et deux fonctions de SciPy (une recherche de racine et un solveur linéaire) qui ne servent
qu'à calculer les **comparateurs a posteriori** de la partie 3. Les algorithmes en ligne sont écrits à la main.

*Ces deux cellules sont fournies : exécutez-les simplement.*

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq, linprog
from IPython.display import display

DATA_DIR = Path.cwd() / "online_learning_bike_sharing_data"
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

PALETTE = {
    "teal": "#00897B",
    "skyblue": "#81D4FA",
    "blue": "#42A5F5",
    "darkblue": "#283593",
    "orange": "#FB8C00",
    "purple": "#8E24AA",
}
# Une couleur et un style de trait par modèle (le style aide aussi à l'impression N&B).
COLORS = {
    "Naïf (veille)": PALETTE["skyblue"],
    "Naïf (J-7)": PALETTE["teal"],
    "OGD": PALETTE["blue"],
    "OGD + Ridge": PALETTE["darkblue"],
    "Newsvendor (sans covariable)": PALETTE["orange"],
    "Newsvendor (contextuel)": PALETTE["purple"],
}
STYLES = {
    "Naïf (veille)": ":", "Naïf (J-7)": ":",
    "OGD": "-", "OGD + Ridge": "-.",
    "Newsvendor (sans covariable)": "--", "Newsvendor (contextuel)": "-",
}
plt.rcParams.update({
    "figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "axes.titleweight": "bold", "axes.labelcolor": "#333333", "lines.linewidth": 2.0,
    "axes.prop_cycle": plt.cycler(color=[PALETTE["teal"], PALETTE["blue"], PALETTE["orange"]]),
})
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

In [ ]:
def load_daily_data(data_dir=DATA_DIR, url=DATA_URL):
    """Charge day.csv : cache local s'il existe, sinon téléchargement de l'archive UCI."""
    data_dir.mkdir(parents=True, exist_ok=True)
    csv_path = data_dir / "day.csv"
    if not csv_path.exists():
        print(f"Téléchargement depuis {url} ...")
        with urlopen(url, timeout=60) as response:
            archive = ZipFile(BytesIO(response.read()))
        csv_path.write_bytes(archive.read("day.csv"))  # on ne lit que le membre attendu
    return pd.read_csv(csv_path, parse_dates=["dteday"])


daily = load_daily_data()
print("Données chargées depuis :", DATA_DIR / "day.csv")

# Partie 1 — Statistiques descriptives

**Jeu de données :** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset),
DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894), licence CC BY 4.0. Nous utilisons `day.csv` :
une ligne par jour, du 1er janvier 2011 au 31 décembre 2012. Citation demandée : Hadi Fanaee-T et João Gama (2013),
*Event labeling combining ensemble detectors and background knowledge*, Progress in Artificial Intelligence.

La variable à prévoir est `cnt`, le nombre total de vélos loués dans la journée.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.1 — Charger et afficher</b><br>
Affichez les premières lignes du jeu de données et son format <code>(n_row, n_col)</code>.
</div>

In [ ]:
# À compléter : afficher les premières lignes de `daily` et son format (n_row, n_col).

**Votre réponse :** ...

*Combien de lignes et de colonnes ? En quelles familles peut-on ranger les colonnes ?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.2 — Résumé statistique</b><br>
Pour chaque variable numérique, calculez moyenne, écart-type, minimum, médiane, maximum, nombre de valeurs manquantes et nombre d'observations. Vérifiez aussi que les dates sont uniques et consécutives, et que <code>cnt = casual + registered</code>.
</div>

In [ ]:
def summary_table(frame):
    """
    Résumé statistique des variables numériques.

    Paramètres
    ----------
    frame : pandas.DataFrame
        Données journalières brutes (sortie de load_daily_data).

    Renvoie
    -------
    table : pandas.DataFrame
        Une ligne par variable numérique (sans la colonne `instant`) et les colonnes
        "mean", "std", "min", "median", "max", "n_nan", "n_obs", arrondies à 3 décimales.
    """
    # À compléter
    ...


# À compléter :
# 1) afficher summary_table(daily) ;
# 2) afficher la période couverte (première et dernière date) ;
# 3) vérifier que les dates sont uniques et consécutives (écart d'exactement un jour) ;
# 4) vérifier que cnt = casual + registered pour tous les jours.

**Votre réponse :** ...

*Y a-t-il des valeurs manquantes ? Quel ordre de grandeur pour `cnt` ? Que déduisez-vous de l'identité `cnt = casual + registered` pour le choix des covariables ? Comment lire les moyennes de `holiday` et `workingday` ?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 1.3 — Tracer la série</b><br>
Tracez la série observée <code>cnt</code> en fonction de la date. Qu'observez-vous ?
</div>

In [ ]:
# À compléter : tracer `cnt` en fonction de `dteday`, avec
#   - la série journalière (trait fin, PALETTE["skyblue"]) ;
#   - une moyenne mobile centrée sur 28 jours (PALETTE["darkblue"]), voir le rappel du préambule ;
#   - une ligne verticale au 1er janvier 2012 (PALETTE["orange"], tirets).
# Titre, axes et légende obligatoires. Affichez aussi la date et la valeur du minimum de `cnt`.

**Votre réponse :** ...

*Tendance ? Saisonnalité ? Variabilité ? Valeurs extrêmes ? Pourquoi la moyenne mobile centrée ne doit-elle pas servir à prévoir ?*

### Les covariables de `day.csv` en un coup d'œil

| Colonne | Description |
|:--|:--|
| `instant` | Indice de la ligne, de 1 à 731. |
| `dteday` | Date calendaire de l'observation. |
| `season` | Saison : 1 = hiver, 2 = printemps, 3 = été, 4 = automne. |
| `yr` | Année : 0 = 2011, 1 = 2012. |
| `mnth` | Mois, de 1 à 12. |
| `holiday` | 1 si le jour est férié, 0 sinon. |
| `weekday` | Jour de la semaine, de 0 (dimanche) à 6 (samedi). |
| `workingday` | 1 si le jour n'est ni un week-end ni un férié, 0 sinon. |
| `weathersit` | Situation météo, de 1 (dégagé) à 4 (forte pluie, neige, orage). |
| `temp` | Température en °C, divisée par 41 (le maximum). |
| `atemp` | Température ressentie en °C, divisée par 50 (le maximum). |
| `hum` | Humidité relative, divisée par 100. |
| `windspeed` | Vitesse du vent, divisée par 67 (le maximum). |
| `casual` | Nombre de locations par des usagers occasionnels (non abonnés). |
| `registered` | Nombre de locations par des abonnés. |
| `cnt` | Total des locations, `casual` + `registered` : **la cible**. |

# Partie 2 — Apprentissage en ligne

## 2.1 Protocole et regret

On joue un jeu répété contre la nature. À chaque jour $t = 1, 2, \dots, T$ :

1. on observe les covariables $\mathbf x_t \in \mathbb R^d$, toutes connues **avant** la décision ;
2. on choisit un paramètre $\theta_t \in \mathcal K$ et on prédit $\widehat y_t = \mathbf x_t^\top \theta_t$ (ou on commande une quantité $q_t$) ;
3. la demande $y_t$ est révélée et on paie la perte $\ell_t(\theta_t)$ ;
4. on corrige : $\theta_{t+1} = \Pi_{\mathcal K}\big(\theta_t - \eta_t\, g_t\big)$ avec $g_t \in \partial \ell_t(\theta_t)$.

C'est l'**Online Gradient Descent projeté** (OGD). Pour la boule $\mathcal K = \{\theta : \|\theta\|_2 \le B\}$, la projection est explicite :
$\Pi_{\mathcal K}(z) = z \cdot \min\{1,\ B/\|z\|_2\}$.

On mesure la qualité par le **regret** : l'écart entre notre perte cumulée et celle du meilleur paramètre **fixe** choisi après coup,
$$
R_T \;=\; \sum_{t=1}^T \ell_t(\theta_t) \;-\; \min_{u \in \mathcal K} \sum_{t=1}^T \ell_t(u).
$$
Aucune hypothèse probabiliste n'est faite sur $(y_t)$ : la suite peut être quelconque, voire adversariale.
Un algorithme est *sans regret* si $R_T/T \to 0$ : en moyenne, il fait asymptotiquement aussi bien que le meilleur choix fixe.

## 2.2 Deux familles de modèles

**(i) Perte quadratique : OGD et OGD + Ridge** (slides 53–59).

| | Perte | Gradient | Mise à jour | Garantie |
|:--|:--|:--|:--|:--|
| OGD | $\ell_t(\theta) = \tfrac12(\mathbf x_t^\top\theta - y_t)^2$ | $e_t\,\mathbf x_t$, avec $e_t = \widehat y_t - y_t$ | $\Pi_{\mathcal K}(\theta_t - \eta\, e_t \mathbf x_t)$ | $R_T \le D\,G\,\sqrt T$ pour $\eta = \frac{D}{G\sqrt T}$ |
| OGD + Ridge | $\widetilde\ell_t(\theta) = \ell_t(\theta) + \tfrac{\lambda}{2}\|\theta\|_2^2$ | $e_t\,\mathbf x_t + \lambda\theta$ | $\Pi_{\mathcal K}\big((1-\eta_t\lambda)\theta_t - \eta_t\, e_t \mathbf x_t\big)$ | $\widetilde R_T \le \frac{G_\lambda^2}{2\lambda}(1+\log T)$ pour $\eta_t = \frac{1}{\lambda t}$ |

avec $D = 2B$ (diamètre de $\mathcal K$), $G = (RB+Y)R$ et $G_\lambda = G + \lambda B$.

- **OGD :** le résidu fixe le signe et l'amplitude de la correction, $\mathbf x_t$ en donne la direction. La perte est convexe
  mais sa hessienne $\mathbf x_t\mathbf x_t^\top$ est de rang 1, donc pas de forte convexité et un regret en $\sqrt T$.
- **Ridge :** la hessienne devient $\mathbf x_t\mathbf x_t^\top + \lambda I_d \succeq \lambda I_d$. La perte est $\lambda$-fortement convexe
  dans **toutes** les directions, d'où un regret logarithmique. Attention : cette garantie porte sur la perte **régularisée**
  $\widetilde\ell_t$, pas sur l'erreur quadratique seule.

**(ii) Perte newsvendor : commander avant de connaître la demande** (slides 60–64).

On commande $q_t \in [0, Q_{\max}]$ avant d'observer $y_t$. Chaque vélo en trop coûte $c_h$, chaque vélo manquant coûte $c_b$ :
$$
\ell_t(q) = c_h\,(q - y_t)_+ + c_b\,(y_t - q)_+ = (c_b + c_h)\,\rho_\tau(y_t - q),
\qquad \tau = \frac{c_b}{c_b + c_h},
$$
où $\rho_\tau(r) = \tau r_+ + (1-\tau)(-r)_+$ est la perte *pinball*. Un sous-gradient est $g_t = c_h$ en cas d'excédent ($q_t > y_t$),
$g_t = -c_b$ en cas de rupture ($q_t < y_t$) et $0$ en cas d'égalité. D'où la règle : **rupture → on monte, excédent → on baisse**,
de façon asymétrique. Avec $\eta = Q_{\max}/(G\sqrt T)$ et $G = \max(c_b, c_h)$, on a $R_T \le Q_{\max}\, G\, \sqrt T$.

**Résultat clé :** pour une demande de loi $F$, le coût espéré est minimal au **quantile critique** $q^\star = F^{-1}(\tau)$.
Sur un échantillon, la meilleure commande constante est le $\tau$-quantile empirique. Nous le vérifierons numériquement.

Le jeu de données ne fournit **pas** de coûts : nous **posons** $c_b = 3$ et $c_h = 1$ (un vélo manquant fait perdre une location,
trois fois plus coûteuse qu'un vélo immobilisé), soit $\tau = 0{,}75$, comme dans le cours. Nous testerons aussi une
version **contextuelle** $q_t = \mathbf x_t^\top\theta_t$, qui apprend un quantile conditionnel avec le même sous-gradient multiplié par $\mathbf x_t$.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.1 — Les hypothèses s'appliquent-elles ?</b><br>
Les garanties de regret reposent sur des hypothèses. Pour chacune, demandez-vous <b>pourquoi</b> elle sert dans la preuve et <b>comment</b> la garantir sur nos données : le domaine $\mathcal K$ est-il convexe et compact ? Les covariables sont-elles bornées, $\|\mathbf x_t\|_2 \le R$ ? La cible est-elle bornée, $|y_t| \le Y$ ? Les pertes sont-elles convexes ? Quelles hypothèses ne sont <b>pas</b> nécessaires ?
</div>

**Votre réponse :** ...

*Pour chaque hypothèse : son rôle dans la preuve, et comment vous la garantissez ici. Quelles hypothèses ne sont pas nécessaires ?*

In [ ]:
# Choix déclarés a priori (fournis, ne pas modifier)
P = 7                      # nombre de retards de la demande utilisés comme covariables
TARGET_SCALE = 10_000.0    # échelle déclarée a priori : y = cnt / 10 000
B = 2.0                    # rayon de K = {θ : ||θ||₂ ≤ B}
C_B, C_H = 3.0, 1.0        # coûts posés : rupture 3× plus chère qu'un excédent
TAU = C_B / (C_B + C_H)    # quantile critique
Q_MAX = 1.0                # capacité newsvendor : 10 000 vélos en unités normalisées

# À compléter : constantes des théorèmes, déduites de la construction des covariables (Question 2.2)
R = ...       # borne de ||x_t||₂
Y = ...       # borne de |y_t|
D = ...       # diamètre de K
G_SQ = ...    # borne du gradient de la perte quadratique sur K
G_NV = ...    # borne du sous-gradient newsvendor, sans covariable
G_CTX = ...   # borne du sous-gradient newsvendor, contextuel


def theory_table(T):
    """
    Pas théoriques et bornes de regret moyen pour un horizon T.

    Paramètres
    ----------
    T : int
        Horizon (nombre de tours).

    Renvoie
    -------
    table : pandas.DataFrame
        Lignes "OGD (quadratique)", "Newsvendor (sans covariable)", "Newsvendor (contextuel)" ;
        colonnes "D", "G", "η* = D/(G√T)" et "borne R_T/T = DG/√T".
    """
    # À compléter
    ...


# À compléter :
# 1) afficher R, Y, B et τ, et vérifier a posteriori que max(cnt) / TARGET_SCALE ≤ Y ;
# 2) afficher theory_table(T=365) ;
# 3) pour λ ∈ {0.01, 0.1}, calculer la borne ridge G_λ²(1 + log T)/(2λT) avec G_λ = G_SQ + λB et T = 365.

**Votre réponse :** ...

*Commentez l'ordre de grandeur de ces bornes face à des pertes quadratiques de l'ordre de 10⁻³. Que garantissent-elles vraiment ?*

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.2 — Préparer les covariables</b><br>
Construisez, pour chaque jour $t$, un vecteur $\mathbf x_t = (1, \mathbf z_t, y_{t-1}, \dots, y_{t-p})$ qui ne contient que de l'information disponible <b>avant</b> d'observer $y_t$. Ici $\mathbf z_t$ regroupe les variables de calendrier connues à l'avance. Le paramètre appris est $\theta_t$, et l'on prédit $\widehat y_t = \mathbf x_t^\top\theta_t$. Quelles colonnes faut-il exclure, et pourquoi ? Vérifiez numériquement que $\|\mathbf x_t\|_2 \le R$ et que les covariables sont causales.
</div>

**Spécification à respecter.** $\mathbf x_t$ contient, dans cet ordre : `intercept` (= 1), `workingday`, `holiday`,
`sin_annual` $=\sin(2\pi j_t/365{,}25)$, `cos_annual` $=\cos(2\pi j_t/365{,}25)$ où $j_t$ est le jour de l'année,
puis les retards `y_lag1`, …, `y_lag7` de $y = \texttt{cnt}/10\,000$. Les jours sans historique complet sont retirés.

**Votre réponse :** ...

*Pourquoi ces covariables ? Quelles colonnes de `day.csv` excluez-vous, et pourquoi chacune ? Combien de jours restent utilisables ?*

In [ ]:
def build_features(frame, p=P, scale=TARGET_SCALE):
    """
    Construit les covariables strictement causales et la cible normalisée.

    Paramètres
    ----------
    frame : pandas.DataFrame
        Données journalières brutes, triées par date.
    p : int, défaut P
        Nombre de retards de la demande.
    scale : float, défaut TARGET_SCALE
        Échelle déclarée : y_t = cnt_t / scale.

    Renvoie
    -------
    X : numpy.ndarray, forme (n, 5 + p)
        Lignes x_t = (1, workingday, holiday, sin_annual, cos_annual, y_lag1, ..., y_lag{p}).
    y : numpy.ndarray, forme (n,)
        Cible normalisée, alignée sur X.
    dates : pandas.Series, longueur n
        Dates des lignes conservées (index remis à zéro).
    feature_names : list of str
        Noms des colonnes de X, dans l'ordre.

    Notes
    -----
    Le retard k du jour t est la demande observée k jours AVANT t. Les p premiers jours,
    dont l'historique est incomplet, sont retirés. Aucune information du jour t ou du futur.
    """
    # À compléter
    ...


X, y, dates, feature_names = build_features(daily)
# À compléter : afficher les formes de X et y, la période couverte, et les 5 premières lignes de X avec leurs noms.

In [ ]:
# À compléter — Vérification 1 : max_t ||x_t||₂ ≤ R et max_t |y_t| ≤ Y.

# À compléter — Vérification 2 (causalité) : copiez `daily`, ajoutez 5 000 à `cnt` au jour t0 = 300, reconstruisez
# les covariables, puis vérifiez avec des `assert` que :
#   - les lignes de X jusqu'au jour t0 inclus sont inchangées (attention au décalage de P lignes) ;
#   - la ligne du lendemain change (via y_lag1).

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 2.3 — Train puis test, en prédisant avant d'apprendre</b><br>
Découpez chronologiquement : <b>train = 2011</b>, <b>test = 2012</b>. Écrivez une boucle en ligne qui, chaque jour, <b>prédit d'abord</b> puis met à jour avec $y_t$. Comment choisir les hyperparamètres sans regarder le test ? Le modèle doit-il continuer d'apprendre pendant le test ? Enfin, retracez la série en colorant les fenêtres de train et de test.
</div>

**Votre réponse :** ...

*Énoncez les règles qui rendent l'évaluation honnête : ordre prédire / apprendre, découpage, choix des hyperparamètres, comportement du modèle pendant le test.*

In [ ]:
# À compléter : masque booléen `is_train` (dates < 2012-01-01), puis
# X_train, y_train, dates_train, X_test, y_test, dates_test (index de dates_test remis à zéro).
# Affichez le nombre de jours et la période de chaque fenêtre.


def run_online(model, X, y):
    """
    Boucle en ligne « prédire, puis apprendre ».

    Paramètres
    ----------
    model : objet
        Fournit params() (paramètre courant), predict(x) et update(x, y).
    X : numpy.ndarray, forme (n, d)
    y : numpy.ndarray, forme (n,)

    Renvoie
    -------
    preds : numpy.ndarray, forme (n,)
        preds[t] est calculée AVANT que y[t] ne soit révélé.
    params : numpy.ndarray, forme (n, dim)
        Copie du paramètre qui a servi à prédire chaque jour (copie, pas alias).
    """
    # À compléter
    ...


def squared_loss(pred, y):
    """Perte quadratique ½ (pred − y)², terme à terme (fonctionne sur des tableaux)."""
    # À compléter
    ...


def newsvendor_loss(q, y, c_b=C_B, c_h=C_H):
    """Perte newsvendor c_h (q − y)_+ + c_b (y − q)_+, terme à terme (fonctionne sur des tableaux)."""
    # À compléter
    ...


def select_on_train(make_model, grid, loss):
    """
    Choisit un hyperparamètre sur la seule fenêtre de train.

    Paramètres
    ----------
    make_model : callable
        make_model(h) renvoie un modèle neuf pour l'hyperparamètre h.
    grid : list
        Valeurs candidates.
    loss : callable
        loss(preds, y) renvoie les pertes terme à terme.

    Renvoie
    -------
    best : élément de grid
        Valeur de perte moyenne en ligne minimale sur le train (la première en cas d'égalité).
    scores : pandas.Series
        Perte moyenne en ligne sur le train, indexée par les valeurs de grid.
    """
    # À compléter
    ...


def train_then_test(make_model, h):
    """
    Apprend en ligne sur le train, puis continue sur le test sans réinitialiser le modèle.

    Renvoie
    -------
    preds, params : sorties de run_online sur la fenêtre de test.
    """
    # À compléter
    ...


results = {}  # nom du modèle → {"pred": ..., "params": ..., "hyper": ...} sur le test (noms : voir COLORS)

In [ ]:
# À compléter : retracer la série (en vélos / jour) avec une aire remplie sous la courbe (fill_between)
# couleur PALETTE["teal"] sur la fenêtre de train et PALETTE["orange"] sur la fenêtre de test,
# une ligne verticale au 1er janvier 2012, l'axe des ordonnées partant de 0, un titre et une légende
# indiquant le nombre de jours de chaque fenêtre.

**Votre réponse :** ...

*Que montre cette figure sur la différence entre les deux fenêtres ? Pourquoi est-ce un argument pour continuer d'apprendre pendant le test ?*

## 2.4 Modèle (i-a) — OGD sur la perte quadratique

On part de $\theta_1 = 0 \in \mathcal K$ avec un pas constant $\eta$, comme dans le théorème. La grille contient le pas théorique
$\eta^\star = D/(G\sqrt{T_{\text{train}}})$ et des pas plus grands. Le théorème raisonne dans le pire cas ; en pratique, un pas plus
grand suit souvent mieux une demande qui évolue.

In [ ]:
def project_ball(theta, radius=B):
    """Projection euclidienne de theta sur K = {θ : ||θ||₂ ≤ radius}."""
    # À compléter
    ...


class OGD:
    """OGD projeté, perte quadratique ½(xᵀθ − y)², pas constant η, départ θ₁ = 0."""

    def __init__(self, eta, d=X.shape[1], radius=B):
        # À compléter : stocker eta et radius, initialiser self.theta
        ...

    def params(self):
        """Renvoie le paramètre courant θ_t."""
        # À compléter
        ...

    def predict(self, x):
        """Renvoie la prédiction x_tᵀθ_t."""
        # À compléter
        ...

    def update(self, x, y):
        """θ_{t+1} = Π_K(θ_t − η e_t x_t), avec e_t = x_tᵀθ_t − y_t."""
        # À compléter
        ...


# À compléter — test de causalité : modifiez y_test[100], relancez run_online(OGD(0.03), ...) et vérifiez avec
# des `assert` que les prédictions jusqu'au jour 100 inclus sont identiques et que les suivantes changent.

eta_star = ...  # à compléter : pas théorique D / (G_SQ √T_train)
OGD_GRID = sorted({round(eta_star, 4), 0.003, 0.01, 0.03, 0.1, 0.3})
# À compléter : best_eta, scores = select_on_train(...) ; afficher scores ;
# puis pred, params = train_then_test(OGD, best_eta) et ranger dans results["OGD"]
# (avec "hyper": f"η = {best_eta}"). Affichez les coefficients finaux avec leurs noms.

**Votre réponse :** ...

*Pourquoi le pas retenu peut-il être plus grand que le pas théorique η* ? Quelles covariables pèsent le plus ? Quel rôle pratique joue la projection ?*

## 2.5 Modèle (i-b) — OGD + Ridge

On applique OGD à $\widetilde\ell_t(\theta) = \tfrac12(\mathbf x_t^\top\theta - y_t)^2 + \tfrac\lambda2\|\theta\|_2^2$. La mise à jour **rétrécit** $\theta$ d'un facteur
$(1 - \eta_t\lambda)$, puis corrige dans la direction $\mathbf x_t$. L'horloge $t$ **n'est pas remise à zéro** au début du test.

**Le pas.** Le cours prend $\eta_t = 1/(\lambda t)$. Pour $\lambda$ petit, les premiers pas sont gigantesques ($\eta_1 = 1/\lambda$) : tant que
$\eta_t\|\mathbf x_t\|^2 > 2$, une correction dépasse sa cible et l'algorithme rebondit sur le bord de $\mathcal K$. Avec $\|\mathbf x_t\|^2 \le R^2$,
cela dure jusqu'à $t \approx R^2/(2\lambda)$ jours : plus de 1 500 jours pour $\lambda = 0{,}003$, soit toute la série. On **plafonne** donc le pas :
$$
\eta_t = \min\Big\{\eta_{\max},\ \frac{1}{\lambda t}\Big\}, \qquad \eta_{\max} = \frac{1}{R^2},
$$
un plafond fixé **a priori** à partir de la borne $R$ de la Question 2.1. Dès que $t \ge R^2/\lambda$, on retrouve exactement le pas du cours ;
le plafond n'ajoute qu'un terme constant à la borne en $\log T$. Pour $\lambda$ petit, la phase plafonnée couvre toute la série :
OGD + Ridge se comporte alors comme OGD à pas $1/R^2$ avec un léger rétrécissement.

On choisit $\lambda$ sur le train avec l'erreur quadratique **non pénalisée**, car c'est la qualité de prévision qui nous intéresse.
Le regret de la partie 3 sera, lui, calculé sur la perte **pénalisée**, celle du théorème.

In [ ]:
class OGDRidge(OGD):
    """
    OGD projeté sur la perte ridge ½(xᵀθ − y)² + λ/2 ||θ||², avec le pas plafonné
    η_t = min(eta_max, 1/(λ t)) et eta_max = 1/R². L'horloge t n'est pas remise à zéro au test.
    """

    def __init__(self, lam, d=X.shape[1], radius=B, eta_max=1 / R**2):
        # À compléter : appeler le constructeur parent, stocker lam et eta_max, initialiser l'horloge t
        ...

    def update(self, x, y):
        """θ_{t+1} = Π_K((1 − η_t λ) θ_t − η_t e_t x_t)."""
        # À compléter
        ...


RIDGE_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
# À compléter : sélection de λ sur le train (perte quadratique NON pénalisée), puis train_then_test ;
# ranger dans results["OGD + Ridge"] avec en plus la clé "lam". Comparez ||θ|| final avec celui d'OGD.

**Votre réponse :** ...

*Que se passe-t-il pour λ petit ? pour λ grand ? (Essayez `eta_max=np.inf`.) La garantie en log T porte-t-elle sur l'erreur de prévision ?*

## 2.6 Modèle (ii) — Perte newsvendor

### Vérification empirique : l'optimum est le quantile $\tau = c_b/(c_b + c_h)$

Sur la fenêtre de train, on calcule le coût moyen $\frac1n\sum_t \ell_t(q)$ d'une commande **constante** $q$ sur une grille fine,
et on compare son minimiseur au $\tau$-quantile empirique de $(y_t)$. On répète l'expérience pour plusieurs couples $(c_b, c_h)$.

In [ ]:
q_grid = np.linspace(0.0, Q_MAX, 4001)
# À compléter :
# 1) coût moyen sur le train d'une commande constante q, pour chaque q de q_grid ;
# 2) son argmin numérique, et le τ-quantile empirique de y_train (np.quantile, method="inverted_cdf") ;
# 3) tracer le coût (en vélos) en fonction de q (en vélos), avec trois lignes verticales :
#    argmin numérique, quantile empirique, moyenne de y_train ;
# 4) refaire la comparaison argmin / quantile pour (c_b, c_h) ∈ {(1, 1), (1, 3), (3, 1), (9, 1)} et l'afficher en tableau.

**Votre réponse :** ...

*L'argmin coïncide-t-il avec le quantile ? Pourquoi peut-il y avoir tout un intervalle de minimiseurs ? Que retrouve-t-on pour c_b = c_h ? Commander la moyenne est-il optimal ?*

### OGD newsvendor sans covariable

C'est l'algorithme des slides 63–64 : $q_{t+1} = \Pi_{[0, Q_{\max}]}(q_t - \eta\, g_t)$. Aucune covariable n'est utilisée, mais la commande
$q_t$ **bouge chaque jour** : rupture, on monte de $\eta c_b$ ; excédent, on baisse de $\eta c_h$. Le **comparateur**, lui, est une seule commande
constante choisie a posteriori : le quantile empirique écrêté.

In [ ]:
class NewsvendorOGD:
    """Commande q_t ∈ [0, Q_max] sans covariable, apprise par sous-gradient projeté (départ q₀)."""

    def __init__(self, eta, q0=0.0, q_max=Q_MAX):
        # À compléter
        ...

    def params(self):
        """Renvoie np.array([q_t])."""
        # À compléter
        ...

    def predict(self, x):
        """Renvoie q_t (ignore les covariables)."""
        # À compléter
        ...

    def update(self, x, y):
        """g_t = c_h si excédent, −c_b si rupture, 0 si égalité ; q_{t+1} = Π_[0, Q_max](q_t − η g_t)."""
        # À compléter
        ...


eta_nv_star = ...  # à compléter : Q_MAX / (G_NV √T_train)
NV_GRID = sorted({round(eta_nv_star, 4), 0.001, 0.003, 0.01, 0.03})
# À compléter : sélection sur le train avec newsvendor_loss, train_then_test, results["Newsvendor (sans covariable)"],
# et la fréquence des jours sans rupture P(y_t ≤ q_t) sur 2012, à comparer à τ.

### OGD newsvendor contextuel : un quantile conditionnel

On remplace la commande sans covariable par $q_t = \mathbf x_t^\top\theta_t$, avec $\theta_t \in \mathcal K$. Le sous-gradient devient
$g_t = \big(c_h\,\mathbf 1\{q_t > y_t\} - c_b\,\mathbf 1\{q_t < y_t\}\big)\,\mathbf x_t$. On apprend ainsi un **quantile conditionnel**
de niveau $\tau$ qui suit le calendrier et la demande récente.

*Remarque.* Rien n'impose $q_t \in [0, Q_{\max}]$ pour un modèle linéaire. On le vérifie a posteriori ; en déploiement, on écrêterait la commande.

In [ ]:
class ContextualNewsvendorOGD(OGD):
    """q_t = x_tᵀθ_t ; sous-gradient (c_h 1{q_t > y_t} − c_b 1{q_t < y_t}) x_t ; projection sur K."""

    def update(self, x, y):
        # À compléter
        ...


CTX_GRID = sorted({round(D / (G_CTX * np.sqrt(len(y_train))), 4), 0.0003, 0.001, 0.003, 0.01})
# À compléter : sélection, train_then_test, results["Newsvendor (contextuel)"], P(y_t ≤ q_t) sur 2012,
# et la part des commandes hors de [0, Q_max].

**Votre réponse :** ...

*Les deux politiques newsvendor couvrent-elles environ τ des jours ? Qu'apporte le contexte ?*

## 2.7 Références naïves

Un algorithme n'a de valeur que comparé à une référence simple. On ajoute deux prédicteurs qui n'apprennent rien :
la **veille** ($\widehat y_t = y_{t-1}$) et la **semaine précédente** ($\widehat y_t = y_{t-7}$). La veille correspond au paramètre fixe
$\theta = e_{\texttt{y\_lag1}}$, de norme $1 \le B$ : c'est un point de $\mathcal K$, donc un concurrent légitime du comparateur.

In [ ]:
class LagBaseline:
    """Prédit la valeur observée k jours plus tôt (colonne y_lag{k} de x_t), sans apprentissage."""

    def __init__(self, k):
        # À compléter : retrouver l'indice de la colonne f"y_lag{k}" dans feature_names
        ...

    def params(self):
        """Aucun paramètre : renvoie np.zeros(0)."""
        # À compléter
        ...

    def predict(self, x):
        # À compléter
        ...

    def update(self, x, y):
        pass


# À compléter : exécuter les deux références sur le test et les ranger dans results["Naïf (veille)"] et results["Naïf (J-7)"]
# (avec "hyper": "—").

ORDER = ["Naïf (veille)", "Naïf (J-7)", "OGD", "OGD + Ridge", "Newsvendor (sans covariable)", "Newsvendor (contextuel)"]
results = {name: results[name] for name in ORDER}

# À compléter : tracer sur 2012 la série observée (gris clair) et les prédictions d'OGD, d'OGD + Ridge
# et du newsvendor contextuel (couleurs COLORS, styles STYLES).

# Partie 3 — Métriques finales

Toutes les métriques portent sur les **366 jours de 2012**, avec les prédictions faites avant chaque révélation.
Les erreurs sont reconverties en **vélos par jour**. Le biais est noté *prédit − observé* : positif, on surestime.
On ajoute le coût newsvendor et la couverture $\mathbb P(y_t \le \widehat y_t)$, qui doit valoir environ $\tau = 0{,}75$ pour un bon quantile.

In [ ]:
def final_metrics(pred, y):
    """
    Métriques de test, erreurs reconverties en vélos (multiplier par TARGET_SCALE).

    Renvoie
    -------
    metrics : dict
        "RMSE", "MAE", "Biais (prédit − observé)", "Coût newsvendor / jour" et "Couverture P(y ≤ ŷ)".
    """
    # À compléter
    ...


# À compléter : `scorecard`, un DataFrame avec une ligne par modèle (ordre de results), les métriques ci-dessus
# et une première colonne "hyperparamètre". Affichez-le.

In [ ]:
# À compléter : deux barcharts horizontaux côte à côte, un par modèle et dans la couleur de COLORS :
#   à gauche le RMSE sur 2012, à droite le coût newsvendor / jour. Valeur affichée au bout de chaque barre.

<div style="border-left:4px solid #FB8C00;padding:8px 14px;background:#FFF8E1">
<b style="color:#283593">Question 3.1 — Le RMSE suffit-il à classer les modèles ?</b><br>
Les modèles newsvendor n'ont pas le RMSE le plus bas. Est-ce un échec ? À quel besoin répond chaque famille de modèles, et avec quelle métrique faut-il juger chacune ? Comparez aussi OGD aux références naïves.
</div>

**Votre réponse :** ...

## Regret moyen au cours du temps

Pour chaque préfixe de $n$ jours de test, on calcule
$$
\frac{R_n}{n} = \frac1n\Big(\sum_{j=1}^n \ell_j(\theta_j) - \min_{u \in \mathcal K}\sum_{j=1}^n \ell_j(u)\Big),
$$
**chaque modèle avec sa propre perte**, c'est-à-dire celle de son théorème. Le comparateur est **un seul** paramètre fixe par préfixe, recalculé à chaque $n$.

- **Quadratique (et ridge) :** $\min_{\|u\|\le B} \tfrac12 u^\top A u - b^\top u$ avec $A = \sum \mathbf x_j\mathbf x_j^\top$ (plus $n\lambda I$ pour le ridge)
  et $b = \sum y_j \mathbf x_j$. Si la solution des moindres carrés est dans la boule, c'est elle. Sinon, conditions KKT :
  $(A + \mu I)u = b$ avec $\|u\| = B$ et $\mu > 0$, où l'on cherche $\mu$ par recherche de racine.
- **Newsvendor sans covariable :** le comparateur est une commande constante ; le coût est linéaire par morceaux, donc l'optimum est atteint en $0$, en $Q_{\max}$ ou en une valeur observée.
- **Newsvendor contextuel :** régression quantile, résolue par un programme linéaire. On vérifie que la solution est dans la boule.
  Si elle n'y est pas, le comparateur calculé est trop bon et la courbe surestime le regret : c'est une borne supérieure prudente.

In [ ]:
def constrained_quadratic_min(A, b, radius=B):
    """
    argmin_{||u||₂ ≤ radius} ½ uᵀAu − bᵀu, pour A symétrique semi-définie positive.

    Indications : diagonalisez A = V diag(w) Vᵀ. Si la solution de norme minimale de Au = b existe et est dans la boule,
    c'est la réponse. Sinon, cherchez μ > 0 tel que ||(A + μI)⁻¹ b|| = radius (brentq), puis u = (A + μI)⁻¹ b.
    """
    # À compléter
    ...


def avg_regret_quadratic(X, y, pred, params=None, lam=0.0):
    """
    Regret moyen R_n / n à chaque préfixe n, pour la perte quadratique (lam = 0) ou ridge (lam > 0).

    Le comparateur du préfixe n minimise sur K la somme des pertes des n premiers jours :
    ½ uᵀA_n u − b_nᵀu + ½ Σ y_j², avec A_n = Σ x_j x_jᵀ (+ nλI pour le ridge) et b_n = Σ y_j x_j.
    Pour le ridge, la perte de l'algorithme inclut λ/2 ||θ_j||² (params[j] = θ_j).

    Renvoie
    -------
    out : numpy.ndarray, forme (n,)
    """
    # À compléter
    ...


def avg_regret_newsvendor_constant(y, q):
    """
    R_n / n contre la meilleure commande constante dans [0, Q_max].
    Indication : le coût est linéaire par morceaux, l'optimum est en 0, en Q_max ou en une valeur observée.
    """
    # À compléter
    ...


def avg_regret_newsvendor_contextual(X, y, q):
    """
    R_n / n contre la meilleure régression quantile linéaire fixe, résolue par programme linéaire :
    min c_b Σ u⁺ + c_h Σ u⁻ sous x_jᵀθ + u⁺_j − u⁻_j = y_j, u⁺, u⁻ ≥ 0 (scipy.optimize.linprog, method="highs").

    Renvoie
    -------
    out : numpy.ndarray, forme (n,)
    inside : numpy.ndarray de booléens, forme (n,)
        True si la solution θ* du préfixe vérifie ||θ*|| ≤ B (sinon la courbe surestime le regret).
    """
    # À compléter
    ...


# À compléter : dictionnaire `avg_regret` (nom → courbe) pour "Naïf (veille)", "OGD", "OGD + Ridge" (perte ridge),
# "Newsvendor (sans covariable)" et "Newsvendor (contextuel)". Affichez la part des préfixes où le comparateur
# contextuel est dans K.

In [ ]:
# À compléter : deux panneaux côte à côte, regret moyen R_n / n en fonction de la date de test, avec une ligne horizontale en 0.
#   gauche : Naïf (veille), OGD, OGD + Ridge (préciser « perte ridge » dans la légende) ;
#   droite : les deux modèles newsvendor.
# Affichez aussi le regret moyen final de chaque modèle.

**Votre réponse :** ...

*Le regret peut-il être négatif ? Pourquoi celui du newsvendor sans covariable l'est-il autant ? Pourquoi le comparateur dépend-il de la classe de modèles ? Un regret moyen faible implique-t-il une erreur faible ? Que peut prouver une courbe finie ?*

## Conclusion

La cellule ci-dessous résume **vos** résultats chiffrés. Les messages qui suivent ne dépendent pas des chiffres exacts.

In [ ]:
best_rmse = scorecard["RMSE"].astype(float).idxmin()
best_cost = scorecard["Coût newsvendor / jour"].astype(float).idxmin()
ogd_vs_naive = scorecard.loc["OGD", "RMSE"] / scorecard.loc["Naïf (veille)", "RMSE"] - 1
print(f"• Meilleur RMSE sur 2012 : {best_rmse} ({scorecard.loc[best_rmse, 'RMSE']:,.0f} vélos / jour).")
print(f"• OGD contre la veille : RMSE {ogd_vs_naive:+.1%}.")
print(f"• Meilleur coût newsvendor : {best_cost} ({scorecard.loc[best_cost, 'Coût newsvendor / jour']:,.0f} / jour).")
print(f"• Couverture P(y ≤ q) : sans covariable = {scorecard.loc['Newsvendor (sans covariable)', 'Couverture P(y ≤ ŷ)']:.2f}, "
      f"contextuel = {scorecard.loc['Newsvendor (contextuel)', 'Couverture P(y ≤ ŷ)']:.2f} (cible {TAU}).")
print("• Regret moyen final : " + ", ".join(f"{k} {v[-1]:+.4f}" for k, v in avg_regret.items()) + ".")

**Votre conclusion :** ...

*En cinq points au plus : ce que la perte change à la décision ; ce qui est commun à tous les algorithmes ; comment vous avez
obtenu les hypothèses des théorèmes ; ce qui rend le protocole honnête ; les limites du regret statique sur cette série.*